# Resume Screening Prediction Training

This notebook covers the exploratory data analysis and model training for the Resume Intelligence Pro project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

## 1. Load Data

In [ ]:
df = pd.read_csv('../dataset/AI_Resume_Screening.csv')
df.head()

## 2. Preprocessing
### 2.1 Handle Multi-label Skills

In [ ]:
df['Skills_List'] = df['Skills'].apply(lambda x: [s.strip() for s in x.split(',')] if isinstance(x, str) else [])
mlb = MultiLabelBinarizer()
skills_encoded = mlb.fit_transform(df['Skills_List'])
skills_df = pd.DataFrame(skills_encoded, columns=mlb.classes_)
skills_df.head()

### 2.2 Encode Categorical Features

In [ ]:
le_ed = LabelEncoder()
df['Education_Encoded'] = le_ed.fit_transform(df['Education'])

le_job = LabelEncoder()
df['Job_Role_Encoded'] = le_job.fit_transform(df['Job Role'])

le_target = LabelEncoder()
df['Decision_Encoded'] = le_target.fit_transform(df['Recruiter Decision'])

## 3. Training & Evaluation

In [ ]:
numeric_cols = ['Experience (Years)', 'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)', 'Education_Encoded', 'Job_Role_Encoded']
X_numeric = df[numeric_cols]
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_numeric), columns=numeric_cols)

X = pd.concat([X_scaled, skills_df], axis=1)
y = df['Decision_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

## 4. Visualization

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='Recruiter Decision', palette='viridis')
plt.title('Hiring Bias Distribution')
plt.show()